# PID Tuning

Finds PID gains for Constant Resistance / Constant Power mode from logged Houston CSVs.
All logic lives in `pid_tuning.py`; this notebook only calls it.

1. Run **Setup** (mounts Drive, fetches `pid_tuning.py`).
2. Fill in **Settings** and run **Tune**.
3. Paste the printed lines into `main.c`, or type the same values into Houston's Live Data panel (`Res Kp` ...).

## Setup

In [ ]:
import os, glob
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
DATA = '/content/drive/MyDrive/WEC_Electronics/Data/'

# Fetch pid_tuning.py from GitHub; fall back to a manual upload if it is not pushed yet.
!wget -q -O pid_tuning.py https://raw.githubusercontent.com/PSU-CWC/Graph/main/pid_tuning.py || rm -f pid_tuning.py
if not os.path.exists('pid_tuning.py') or os.path.getsize('pid_tuning.py') == 0:
    from google.colab import files
    print('Upload pid_tuning.py')
    files.upload()

import importlib, pid_tuning
importlib.reload(pid_tuning)
from pid_tuning import *
_self_check()

## Settings

- `FILES`: one path or a list. Paths are relative to `DATA`; wildcards work (`'testing_2-15-2026/*.csv'`).
- `Y_COL`: `'Load Resistance'` (constant resistance) or `'Load Power'` (constant power).
- `SETPOINT`: target resistance (Ohm) or power (W).
- `START`, `END`: time window in **seconds**, applied to every file. `None` = to the end. Pick it from the plot below.
- `TS`: control loop period in seconds. `None` = median interval between DAC updates (Houston rows are GUI frames, not loop iterations).

In [ ]:
FILES = ['constant_resistance/pid_data3.csv']
Y_COL = 'Load Resistance'
SETPOINT = 500
START, END = 0, None
TS = None
MIN_R2 = 0.8

## Look at the data first (pick START / END here)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
paths = sorted(p for f in FILES for p in glob.glob(DATA + f))
print(len(paths), 'file(s)')
for p in paths:
    df = pd.read_csv(p)
    sec = (df['Time'] - df['Time'].iloc[0]) / 1000
    u_col = next(c for c in DAC_COLUMNS if c in df.columns)
    fig, ax = plt.subplots(2, 1, sharex=True, figsize=(12, 5))
    ax[0].plot(sec, df[Y_COL]); ax[0].set_ylabel(Y_COL); ax[0].set_title(os.path.basename(p))
    ax[1].plot(sec, df[u_col]); ax[1].set_ylabel(u_col); ax[1].set_xlabel('time (s)')
    plt.tight_layout(); plt.show()

## Tune

In [ ]:
gains = run(paths, Y_COL, SETPOINT, start=START, end=END, ts=TS, min_r2=MIN_R2)